<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Spark DataFrames and SQL</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 2.3 Demo - Reading and Writing Data with DataFrames
### Duration - 14 Minutes

This demonstration introduces Spark DataFrames by reading data from an external source into a DataFrame and then writing the DataFrame out to another location.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Read external data into a DataFrame
- Create and inspect DataFrame schemas
- Write the contents of a DataFrame to a specified location

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations explore the Spark UI and cluster internals, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run, sets your default catalog, and sets the default schema to `intro_spark`.

In [0]:
%run ../Includes/Classroom-Setup-2

In [0]:
%python
# Capture the Catalog Name
catalog = spark.catalog.currentCatalog()
source_path = f"/Volumes/{catalog}/intro_spark/source_files/customers"

print(f"Reading source files from: {source_path}")

## B. Reading Data

We will read the same directory of CSV files three ways: with schema inference, with an explicit `StructType` schema, and with a DDL string schema.

### B1. Schema Inference

Let's start by creating a DataFrame from a directory of CSV files, asking Spark to infer the schema from the data.

In [0]:
%python
# Read CSV with Schema Inference
customers_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path))

In [0]:
%python
# Show the Inferred Schema
customers_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128221;</span>
<div>
<strong style="color: #4a148c; font-size: 1.1em">Headers and Column Names</strong>
<p style="margin: 8px 0 0 0; color: #333">If the dataset has a header row, column names are inferred from it. Without a header, columns are named <code>_c0</code>, <code>_c1</code>, and so on, and must be renamed manually.</p>
</div>
</div>
</div>

In [0]:
%python
# Print the Schema in Linear Format
# Alternatively, print the schema as a single StructType expression
print(customers_df.schema)

### B2. The display() Function

The `display()` function is available in Databricks runtimes for use within notebooks.

`display()` triggers a Spark action, which returns results for a DataFrame in a formatted tabular result window. As this is intended for visual inspection only, results are limited to 10,000 rows or 2 MB, whichever is less. Additional features of the `display()` function include:

- Ability to sort or filter data in the result set
- Ability to download result data to a CSV file
- Ability to profile data using the **+** -> **Data Profile** option
- Ability to visualize data using the **+** -> **Visualization** option

In [0]:
%python
# Preview the Data with display()
display(customers_df)

### B3. Explicitly Defining a Schema

Let's load the same dataset into a DataFrame, this time explicitly defining the schema with `StructType` and `StructField` objects. This avoids the extra pass over the data that inference requires and guarantees the types you expect.

In [0]:
%python
# Read CSV with an Explicit StructType Schema
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Define explicit schema using StructType
customer_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("nation_key", LongType(), True),
    StructField("phone", StringType(), True),
    StructField("account_balance", DoubleType(), True),
    StructField("market_segment", StringType(), True),
    StructField("comment", StringType(), True),
])

# Read the customers CSV files with the explicit schema
customers_structtype_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load(source_path))

In [0]:
%python
# Examine the Explicit Schema
customers_structtype_df.printSchema()

### B4. Using a DDL Schema

Now let's define the same schema using a DDL string, which is easier to read and write.

In [0]:
%python
# Define a DDL Schema
ddl_schema = '''
  customer_id BIGINT NOT NULL,
  name STRING,
  address STRING,
  nation_key BIGINT,
  phone STRING,
  account_balance DOUBLE,
  market_segment STRING,
  comment STRING
'''

In [0]:
%python
# Read CSV with the DDL Schema
customers_ddl_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(ddl_schema)
    .load(source_path))

customers_ddl_df.printSchema()

## C. Writing Data from DataFrames

Now let's write out the contents of a DataFrame to different output locations.

### C1. Writing to a File System

We can write the contents of our DataFrame to a file system in various formats using the `write` and `save` methods. Here we write Parquet files to the `output` volume created during setup.

In [0]:
%python
# Write the DataFrame as Parquet Files
# Define an output path in the output volume
parquet_output_path = f"/Volumes/{catalog}/intro_spark/output/customers_parquet"

# Write the DataFrame out as Parquet files to a directory
(customers_ddl_df.write
    .format("parquet")
    .mode("overwrite")
    .save(parquet_output_path))

In [0]:
%python
# List the Output Files
display(dbutils.fs.ls(parquet_output_path))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The output is a <em>directory</em> of Parquet part files, not a single file - one file per partition written in parallel by the Executors.</p>
<p style="margin: 8px 0 0 0; color: #333">Parquet is a columnar, self-describing format: the schema travels with the files, so reading them back requires no inference and no DDL.</p>
</div>

### C2. Writing to a Table

We can also save our DataFrame to a table (defined in a catalog and schema in Unity Catalog).

In [0]:
%python
# Save the DataFrame as a Table
# Writing our DataFrame to a new table (this is an action)
customers_ddl_df.write.mode("overwrite").saveAsTable("customers_ddl")

In [0]:
%python
# Save Using the DataFrameWriterV2 API
# Alternatively you can use the writeTo method, which invokes the DataFrameWriterV2
(customers_ddl_df.writeTo(f"{catalog}.intro_spark.customers_ddl")
    .createOrReplace())

# DataFrameWriterV2 also offers options to partition the table, as well as append or overwrite

In [0]:
-- Read Back the Table with SQL
SELECT customer_id, name, market_segment, account_balance
FROM customers_ddl
LIMIT 10;

## Summary

**Reading data into DataFrames:**

- Schema inference can be used for CSV files, at the cost of an extra pass over the data
- The schema can also be explicitly defined (preferred) using `StructType` definitions or a DDL string

**Writing data from DataFrames:**

- DataFrames can be written out to a distributed file system (like a Unity Catalog volume) in formats such as Parquet
- DataFrames can also be written out to tables in Unity Catalog using `saveAsTable` or `writeTo`

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>